In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dataset 06 — Credit Approval — UCI

**Descripción:** Dataset de aprobación de tarjetas de crédito con atributos mezclados (numéricos y categóricos). Los nombres de los atributos y valores han sido cambiados a símbolos por razones de confidencialidad.

**Tarea:** Clasificación binaria — predecir si una solicitud de crédito es aprobada (+) o rechazada (-).

**Características:**
- m = 690 registros
- n = 15 características + 1 clase
- Variable objetivo: `A16` (`+` aprobado, `-` rechazado)
- Valores nulos: codificados como `?` en varias columnas
- Mezcla de variables continuas y categóricas (nombres anonimizados A1-A15)

**URL UCI:** https://archive.ics.uci.edu/ml/datasets/credit+approval

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
print('Librerías cargadas correctamente')

## 1. Carga del Dataset

In [ ]:
columnas = [f'A{i}' for i in range(1,16)] + ['clase']
ruta = '/content/drive/MyDrive/machine learning/datasets/crx.data'
data = pd.read_csv(ruta, names=columnas, na_values='?')
print(f'Dimensiones: {data.shape}')
data.head()

## 2. Revisión de Tipos y Nulos

In [ ]:
print(data.dtypes)
print()
nulos = data.isnull().sum()
print('Valores nulos:')
print(nulos[nulos>0])
print(f'Total: {data.isnull().sum().sum()}')

## 3. Preprocesamiento con Pandas

In [ ]:
df = data.copy()

# Imputar nulos: numéricos con mediana, categóricos con moda
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == object:
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

# Codificar variable objetivo
df['target'] = (df['clase'] == '+').astype(int)

# Codificar categóricas
cols_cat = df.select_dtypes(include=['object']).columns
for col in cols_cat:
    df[col] = pd.Categorical(df[col]).codes

print(f'Nulos restantes: {df.isnull().sum().sum()}')
print(f'\nAprobadas (+): {int((df["target"]==1).sum())} ({(df["target"]==1).mean()*100:.1f}%)')
print(f'Rechazadas (-): {int((df["target"]==0).sum())} ({(df["target"]==0).mean()*100:.1f}%)')

## 4. Estadísticas Descriptivas

In [ ]:
df.describe()

## 5. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['Rechazada (0)','Aprobada (1)'],
            [int((df['target']==0).sum()),int((df['target']==1).sum())],
            color=['tomato','steelblue'],edgecolor='black')
axes[0].set_title('Distribución de Aprobación de Crédito')
axes[0].set_ylabel('Número de solicitudes')

axes[1].hist(data['A2'].dropna(), bins=20, color='green', edgecolor='black')
axes[1].set_title('Distribución de A2 (numérica)')
axes[1].set_xlabel('Valor')
axes[1].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

## 6. División 80/20

In [ ]:
X = df[[col for col in df.columns if col not in ['clase','target']]].to_numpy(dtype=np.float64)
y = df['target'].to_numpy()
np.random.seed(42)
idx = np.random.permutation(len(X))
corte = int(len(X)*0.8)
X_train,X_test = X[idx[:corte]],X[idx[corte:]]
y_train,y_test = y[idx[:corte]],y[idx[corte:]]
print(f'Entrenamiento: {X_train.shape[0]} | Prueba: {X_test.shape[0]}')

## 7. Conclusiones

**Credit Approval** es un dataset pequeño (690 registros) pero clásico para clasificación binaria. Los atributos están anonimizados (A1-A15) por confidencialidad. Contiene valores nulos codificados como `?` que requieren imputación. La distribución de clases está aproximadamente balanceada (~55% aprobadas). Es adecuado para aplicar regresión logística binaria.